In [1]:
import os
import json
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 1. Load BOTH datasets just like in your main RAG pipeline
df_safety = pd.read_csv('../data/food_safety.csv')
df_rolls = pd.read_csv('../data/OneRoll_updated.csv')

df = pd.concat([df_safety, df_rolls], ignore_index=True)
df = df.fillna('')

# 2. Build standardized documents with correct IDs (0 to 69)
documents = []
for idx, row in df.iterrows():
    doc = {col: ('' if pd.isna(row[col]) or str(row[col]).lower() in ['nan', 'none'] else str(row[col])) for col in df.columns}
    doc['id'] = idx
    documents.append(doc)

print(f"Generating ground truth for {len(documents)} total documents...")

prompt_template = """
You emulate a user of our AFC Sushi and food safety assistant application.
Formulate 5 questions this user might ask based on the provided record.
Make the questions specific to this record, complete, and not too short.
Use as fewer words as possible from the record.

The record:
{record_fields}

Provide the output in parsable JSON without using code blocks:
{{"questions": ["question1", "question2", ..., "question5"]}}
""".strip()

def generate_questions(doc):
    # Dynamically format whichever fields exist in the record (handles both food safety & roll columns)
    record_str = "\n".join([f"{k}: {v}" for k, v in doc.items() if k != 'id' and v])
    prompt = prompt_template.format(record_fields=record_str)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

# 3. Loop through all 70 documents
results = {}
for idx, doc in enumerate(tqdm(documents, desc="Generating Questions")):
    try:
        questions_raw = generate_questions(doc)
        questions = json.loads(questions_raw)
        results[idx] = questions['questions']
    except Exception as e:
        print(f"Error on doc {idx}: {e}")

# 4. Compile and save to CSV
final_results = []
for doc_id, questions in results.items():
    for q in questions:
        final_results.append((doc_id, q))

df_results = pd.DataFrame(final_results, columns=['id', 'question'])
os.makedirs('../data', exist_ok=True)
df_results.to_csv('../data/sushi-ground-truth-retrieval.csv', index=False)

print(f"Successfully generated {len(df_results)} questions across {len(documents)} documents and saved to '../data/sushi-ground-truth-retrieval.csv'!")

/Users/pauenpiang/afc-sushi-assistant/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating ground truth for 73 total documents...


Generating Questions: 100%|██████████| 73/73 [01:18<00:00,  1.08s/it]

Successfully generated 365 questions across 73 documents and saved to '../data/sushi-ground-truth-retrieval.csv'!
